# **Trabajo Práctico N° 2 - Aprendizaje Automático II - 2024**

### **Redes Recurrentes**
**PROBLEMA 2 - Shakespear**

---
**Alumno:**


*   **Fontana Gustavo**

**Legajo:**


*   **F-3749/4**


## **Librerías**

In [ ]:
import tensorflow as tf
import numpy as np
import textwrap
import shutil
import gdown
import json
from tensorflow.keras.models import load_model

## **Configuración inicial**

In [ ]:
# Configurar para que TensorFlow utilice la GPU por defecto
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        # Configurar para que TensorFlow asigne memoria dinámicamente
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        # Especificar la GPU por defecto
        logical_gpus = tf.config.experimental.list_logical_devices('GPU')
        print(len(gpus), "Physical GPUs,", len(logical_gpus), "Logical GPUs")
    except RuntimeError as e:
        # Manejar error
        print(e)

1 Physical GPUs, 1 Logical GPUs


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## **Carga del dataset**

**El dataset proporcionado incluye 40000 líneas de distintos escritos de Shakespear.**



In [ ]:
# Descargar el dataset
url = 'https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt'
gdown.download(url, 'shakespear.txt', quiet=True)

'shakespear.txt'

In [ ]:
# Leer el archivo
text = open("/content/shakespear.txt", 'rb').read().decode(encoding='utf-8')

# Cantidad de caracteres en el texto
print(f'Length of text: {len(text)} characters')

Length of text: 1115394 characters


In [ ]:
# Inspeccionar el dataset
print(text[:300])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us


## **Implementación del modelo**

In [ ]:
@tf.keras.utils.register_keras_serializable()
class MyModel(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim, rnn_units):
        super().__init__()
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
        self.lstm = tf.keras.layers.LSTM(rnn_units, return_sequences=True, return_state=True)
        self.dense = tf.keras.layers.Dense(vocab_size)

    def build(self, input_shape):
        """Construye las capas del modelo definiendo las dimensiones de entrada."""
        batch_size, seq_length = input_shape
        self.embedding.build((batch_size, seq_length))
        self.lstm.build((batch_size, seq_length, self.embedding.output_dim))
        self.dense.build((batch_size, seq_length, self.lstm.units))
        self.built = True

    def call(self, inputs, states=None, return_state=False, training=False):
      x = inputs
      x = self.embedding(x, training=training)
      if states is None:
        states = [tf.zeros((tf.shape(x)[0], self.lstm.units)),
                  tf.zeros((tf.shape(x)[0], self.lstm.units))]
      x, state_h, state_c = self.lstm(x, initial_state=states, training=training)
      x = self.dense(x, training=training)

      if return_state:
        return x, [state_h, state_c]
      else:
        return x

## **Modelo Caracter a caracter**

### **Preproceseamiento del texto**

In [ ]:
# Función para separar la secuencia en entrada y salida
def split_input_target(sequence):
    input_text = sequence[:-1]
    target_text = sequence[1:]
    return input_text, target_text

In [ ]:
# Obtener el conjunto de caracteres únicos
vocab_1 = sorted(set(text))

# Crear la capa StringLookup para mapear caracteres a índices num.
ids_from_chars = tf.keras.layers.StringLookup(
    vocabulary=list(vocab_1),
    mask_token=None)

# Crear la capa StringLookup para mapear índices num. a caracteres
chars_from_ids = tf.keras.layers.StringLookup(
    vocabulary=ids_from_chars.get_vocabulary(),
    invert=True,
    mask_token=None)

# Convertir el texto a índices utilizando StringLookup
all_ids_1 = ids_from_chars(tf.strings.unicode_split(text, 'UTF-8'))

# Longitud de las secuencias
seq_length_1 = 100

# Crear las secuencias de entrenamiento
ids_dataset_1 = tf.data.Dataset.from_tensor_slices(all_ids_1)

# Crear las secuencias de entrada y salida
sequences_1 = ids_dataset_1.batch(seq_length_1 + 1, drop_remainder=True)

# Aplicar la función para separar entrada y salida a las secuencias
char2char_dataset = sequences_1.map(split_input_target)

**Proceso:**

*  Se obtienen en primer lugar los caracteres únicos, los cuales con la capa `tf.keras.layers.StringLookup` son mapeados a indices numéricos. De esta manera la red neuronal pueda trabajar con datos numéricos en lugar de texto.

* La capa `tf.keras.layers.StringLookup` con el parámetro `invert` en True, nos permite invertir el mapeo de indices numéricos a caracteres para poder interpretar los resultados generados por el modelo.

*  Se agrupan los indices en secuencias de caracteres, cada secuencia contiene un `seq_length+1` de caracteres.

* La función split_input_target separa las secuencias de caracteres de entrada y salida. Se crean entradas (`seq_length`) y salidas (siguiente carácter).

**Ejemplos de cada paso en el proceso**

In [ ]:
# Información de las variables
print(f'Hay un total de {len(vocab_1)} caracteres únicos.\n')
print(f'Hay un total de {len(char2char_dataset)} secuencias.')

Hay un total de 65 caracteres únicos.

Hay un total de 11043 secuencias.


In [ ]:
# Primeros treinta caracteres del texto
tf.strings.unicode_split(text, 'UTF-8')[:30].numpy()

array([b'F', b'i', b'r', b's', b't', b' ', b'C', b'i', b't', b'i', b'z',
       b'e', b'n', b':', b'\n', b'B', b'e', b'f', b'o', b'r', b'e', b' ',
       b'w', b'e', b' ', b'p', b'r', b'o', b'c', b'e'], dtype=object)

In [ ]:
# Mapeo de caracteres a indices
all_ids_1[:30].numpy()

array([19, 48, 57, 58, 59,  2, 16, 48, 59, 48, 65, 44, 53, 11,  1, 15, 44,
       45, 54, 57, 44,  2, 62, 44,  2, 55, 57, 54, 42, 44])

In [ ]:
# Mapeo de indices a caracteres
example_ids = all_ids_1[:30].numpy().tolist()

chars_from_ids(example_ids)

<tf.Tensor: shape=(30,), dtype=string, numpy=
array([b'F', b'i', b'r', b's', b't', b' ', b'C', b'i', b't', b'i', b'z',
       b'e', b'n', b':', b'\n', b'B', b'e', b'f', b'o', b'r', b'e', b' ',
       b'w', b'e', b' ', b'p', b'r', b'o', b'c', b'e'], dtype=object)>

In [ ]:
# Ejemplo de los primers diez elementos del dataset
for num in ids_dataset_1.take(10):
    print(num.numpy(), end=" ")

19 48 57 58 59 2 16 48 59 48 

In [ ]:
# Ejemplo de una secuencia entrada/salida
for seq in char2char_dataset.take(1):
  print(seq)

(<tf.Tensor: shape=(100,), dtype=int64, numpy=
array([19, 48, 57, 58, 59,  2, 16, 48, 59, 48, 65, 44, 53, 11,  1, 15, 44,
       45, 54, 57, 44,  2, 62, 44,  2, 55, 57, 54, 42, 44, 44, 43,  2, 40,
       53, 64,  2, 45, 60, 57, 59, 47, 44, 57,  7,  2, 47, 44, 40, 57,  2,
       52, 44,  2, 58, 55, 44, 40, 50,  9,  1,  1, 14, 51, 51, 11,  1, 32,
       55, 44, 40, 50,  7,  2, 58, 55, 44, 40, 50,  9,  1,  1, 19, 48, 57,
       58, 59,  2, 16, 48, 59, 48, 65, 44, 53, 11,  1, 38, 54, 60])>, <tf.Tensor: shape=(100,), dtype=int64, numpy=
array([48, 57, 58, 59,  2, 16, 48, 59, 48, 65, 44, 53, 11,  1, 15, 44, 45,
       54, 57, 44,  2, 62, 44,  2, 55, 57, 54, 42, 44, 44, 43,  2, 40, 53,
       64,  2, 45, 60, 57, 59, 47, 44, 57,  7,  2, 47, 44, 40, 57,  2, 52,
       44,  2, 58, 55, 44, 40, 50,  9,  1,  1, 14, 51, 51, 11,  1, 32, 55,
       44, 40, 50,  7,  2, 58, 55, 44, 40, 50,  9,  1,  1, 19, 48, 57, 58,
       59,  2, 16, 48, 59, 48, 65, 44, 53, 11,  1, 38, 54, 60,  2])>)


### **Implementación del modelo Caracter a caracter**

In [ ]:
# Configuración del batch size
BATCH_SIZE = 64
BUFFER_SIZE = 10000
char2char_dataset = (char2char_dataset
                     .shuffle(BUFFER_SIZE)
                     .batch(BATCH_SIZE, drop_remainder=True)
                     .prefetch(tf.data.experimental.AUTOTUNE))

In [ ]:
# Tamaño del vocabulario
vocab_size_1 = len(ids_from_chars.get_vocabulary())

# Dimensión de los embedding
embedding_dim = 256

# Número de unidades RNN
rnn_units = 1024

char2char_model = MyModel(
    vocab_size=vocab_size_1,
    embedding_dim=embedding_dim,
    rnn_units=rnn_units)

char2char_model.build(input_shape=(BATCH_SIZE, vocab_size_1))
char2char_model.summary()

# Compilar el modelo
char2char_model.compile(optimizer=tf.keras.optimizers.Adam(),
                        loss=tf.losses.SparseCategoricalCrossentropy(from_logits=True))

Model: "my_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ (64, 66, 256)               │          16,896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm (LSTM)                          │ ((64, 66, 1024), (64,       │       5,246,976 │
│                                      │ 1024), (64, 1024))          │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (64, 66, 66)                │          67,650 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 5,331,522 (20.34 MB)

 Trainable params: 5,331,522 (20.34 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
EPOCHS = 100
char2char_history = char2char_model.fit(char2char_dataset,
                                  epochs=EPOCHS,
                                  )

Epoch 1/100
172/172 ━━━━━━━━━━━━━━━━━━━━ 13s 66ms/step - loss: 0.1065
Epoch 2/100
172/172 ━━━━━━━━━━━━━━━━━━━━ 13s 68ms/step - loss: 0.0938
Epoch 3/100
172/172 ━━━━━━━━━━━━━━━━━━━━ 14s 70ms/step - loss: 0.0862
Epoch 4/100
172/172 ━━━━━━━━━━━━━━━━━━━━ 14s 70ms/step - loss: 0.0816
Epoch 5/100
172/172 ━━━━━━━━━━━━━━━━━━━━ 14s 72ms/step - loss: 0.0802
Epoch 6/100
172/172 ━━━━━━━━━━━━━━━━━━━━ 15s 74ms/step - loss: 0.0788
Epoch 7/100
172/172 ━━━━━━━━━━━━━━━━━━━━ 15s 75ms/step - loss: 0.0787
Epoch 8/100
172/172 ━━━━━━━━━━━━━━━━━━━━ 14s 73ms/step - loss: 0.0797
Epoch 9/100
172/172 ━━━━━━━━━━━━━━━━━━━━ 20s 72ms/step - loss: 0.0802
Epoch 10/100
172/172 ━━━━━━━━━━━━━━━━━━━━ 14s 74ms/step - loss: 0.0810
Epoch 11/100
172/172 ━━━━━━━━━━━━━━━━━━━━ 21s 74ms/step - loss: 0.0838
Epoch 12/100
172/172 ━━━━━━━━━━━━━━━━━━━━ 20s 75ms/step - loss: 0.1835
Epoch 13/100
172/172 ━━━━━━━━━━━━━━━━━━━━ 15s 75ms/step - loss: 0.8419
Epoch 14/100
172/172 ━━━━━━━━━━━━━━━━━━━━ 14s 74ms/step - loss: 0.4890
Epoch 15/100
17

In [ ]:
# Guardar modelo
char2char_model.save(
    '/content/drive/MyDrive/Aprendizaje 2/TP2/char2char_model.keras')

In [ ]:
# Cargar el modelo guardado
char2char_model = tf.keras.models.load_model(
    '/content/drive/MyDrive/Aprendizaje 2/TP2/char2char_model.keras')

### **Predicción caracter a caracter**

In [ ]:
class OneStep(tf.keras.Model):
  def __init__(self, model, chars_from_ids, ids_from_chars, temperature=1.0):
    super().__init__()
    self.temperature = temperature
    self.model = model
    self.chars_from_ids = chars_from_ids
    self.ids_from_chars = ids_from_chars

  @tf.function
  def generate_one_step(self, inputs, states=None):
    # Convertir las cadenas a identificadores de tokens.
    input_chars = tf.strings.unicode_split(inputs, 'UTF-8')
    input_ids = self.ids_from_chars(input_chars).to_tensor()

    # Ejecutar el modelo.
    # predicted_logits.shape es [batch, char, next_char_logits]
    predicted_logits, states = self.model(inputs=input_ids, states=states,
                                          return_state=True)
    # Usar solo la última predicción.
    predicted_logits = predicted_logits[:, -1, :]
    predicted_logits = predicted_logits / self.temperature

    # Muestrear los logits de salida para generar identificadores de tokens.
    predicted_ids = tf.random.categorical(predicted_logits, num_samples=1)
    predicted_ids = tf.squeeze(predicted_ids, axis=-1)

    # Convertir los identificadores de tokens a caracteres
    predicted_chars = self.chars_from_ids(predicted_ids)

    # Devolver los caracteres y el estado del modelo.
    return predicted_chars, states

In [ ]:
# Función para generar fragmentos de texto
def text_generator(model, temperature, seq_length, initial_imput="Oh Lord", num_fragments=2):
    # Lista para almacenar los fragmentos generados
    generated_fragments = []

    # Crear el modelo de generación con la temperatura específica
    one_step_model = OneStep(model, chars_from_ids, ids_from_chars, temperature=temperature)

    print(f"Temperatura: {temperature}\nLongitud: {seq_length}")

    for i in range(num_fragments):
        states = None
        next_char = tf.constant([initial_imput])
        result = [next_char]

        # Generar texto con la longitud especificada
        for _ in range(seq_length):
            next_char, states = one_step_model.generate_one_step(next_char, states=states)
            result.append(next_char)

        # Convertir a cadena y almacenar en la lista
        text_new = tf.strings.join(result).numpy()[0].decode('utf-8')
        generated_fragments.append((temperature, seq_length, text_new))

        # Mostrar el fragmento generado
        print(f"\nFragmento {i+1}:\n{text_new}\n")

In [ ]:
# Primera generación de texto
text_generator(char2char_model, 1.3, 50, "The truth", 10)

Temperatura: 1.3
Longitud: 50

Fragmento 1:
The truth shall be better thee ready to live:
But by joint-


Fragmento 2:
The truth will be bent their subjects enjoy.

LADY ANNE:
Vi


Fragmento 3:
The truth of conscience take it from
the easiest between th


Fragmento 4:
The truth with such a quarrel daylight in what pity
is my t


Fragmento 5:
The truth with she-give their dired world's valueth
but ric


Fragmento 6:
The truth will be penished in this tune.

BASTHASAR:
I will


Fragmento 7:
The truth will set again with crowns.

BUCKINGHAM:
Why, the


Fragmento 8:
The truth of composion: how shall I do think,
To let him fr


Fragmento 9:
The truth of conscience strike always long,
May think of th


Fragmento 10:
The truth common people had embraced me in
so far from henc



In [ ]:
# Segunda generación de texto
text_generator(char2char_model, 0.5, 50, "The love")

Temperatura: 0.5
Longitud: 50

Fragmento 1:
The love have been determined, to our see
shape of comfort


Fragmento 2:
The love shall be death disgraced thy daughter.
What, will



In [ ]:
# Tercera generación de texto
text_generator(char2char_model, 1, 100)

Temperatura: 1
Longitud: 100

Fragmento 1:
Oh Lord Hastings, let us all good
Would not have the grain. But who comes here?

DUKE OF AUMERLE:
Where is 


Fragmento 2:
Oh Lord Hastings, let us hear him pass.

Second Servingman:
Ay, and his himself, but madam: ha!

AUTOLYCUS:



In [ ]:
# Cuarta generación de texto
text_generator(char2char_model, 1, 100)

Temperatura: 1
Longitud: 100

Fragmento 1:
Oh Lord Hastings, let us hear him pass.

Second Senator:
Noble lord, for I have coming not to such man: we 


Fragmento 2:
Oh Lord Hastings, let us all good,
Were they unwisedous to the second Katean:
The banish'd Auntable dishono



In [ ]:
# Quinta generación de texto
text_generator(char2char_model, 1.5, 200)

Temperatura: 1.5
Longitud: 200

Fragmento 1:
Oh Lord Hastings, when it blows,
Comprison with the aims and envy is dear;
And spurn upon thee, loving maidens.

GLOUCESTER:
Nouried he is been,
And make a deep innatuary Vince of Walk,
Withal, despite of an


Fragmento 2:
Oh Lord Hastings, let us last they serve
To her husband and my shame with Thomas doub:
His cloud is wet, and excuse my born,
Let purse great ordnance on a naple,
Or fell this premedine better 'Gainstly,
That



## **Modelo Palabra a palabra**

### **Preprocesamiento del texto**

In [ ]:
# Función para separar la secuencia en entrada y salida
def split_input_target(sequence):
    input_text = sequence[:-1]
    target_text = sequence[1:]
    return input_text, target_text

In [ ]:
# Dividir considerando saltos de línea como tokens
vocab_2 = text.split()
#vocab_2 = re.findall(r'\S+|\s+', text)

# Crear capa StringLookup con los saltos de línea incluidos
ids_from_words = tf.keras.layers.StringLookup(
    vocabulary=list(set(vocab_2)),
    mask_token=None
)

# Crear la inversa
words_from_ids = tf.keras.layers.StringLookup(
    vocabulary=ids_from_words.get_vocabulary(),
    invert=True,
    mask_token=None
)

# Mapear texto a índices
all_ids_2 = ids_from_words(vocab_2)

# Crear dataset a partir de índices
ids_dataset_2 = tf.data.Dataset.from_tensor_slices(all_ids_2)

# Configurar longitud de secuencias
seq_length_2 = 60

# Crear batches con tamaño seq_length + 1
sequences_2 = ids_dataset_2.batch(seq_length_2 + 1, drop_remainder=True)

# Crear dataset final
word2word_dataset = sequences_2.map(split_input_target)

### **Implementación el modelo palabra a palabra**

In [ ]:
print(word2word_dataset.element_spec)

(TensorSpec(shape=(60,), dtype=tf.int64, name=None), TensorSpec(shape=(60,), dtype=tf.int64, name=None))


In [ ]:
# Configuración de batch size y optimización del dataset
BATCH_SIZE = 64
BUFFER_SIZE = 10000
w2w_dataset = (word2word_dataset
                     .shuffle(BUFFER_SIZE)
                     .batch(BATCH_SIZE, drop_remainder=True)
                     .prefetch(tf.data.experimental.AUTOTUNE))

for input, output in w2w_dataset.take(1):
    print(input.shape, output.shape)

(64, 60) (64, 60)


In [ ]:
# Modelo
vocab_size_2 = len(ids_from_words.get_vocabulary()) #25671
embedding_dim = 256
rnn_units = 1024
input_shape = input.shape

word2word_model = MyModel(
    vocab_size=vocab_size_2,
    embedding_dim=embedding_dim,
    rnn_units=rnn_units
)

word2word_model.build(input_shape=input_shape)
word2word_model.summary()

# Compilación
word2word_model.compile(optimizer=tf.keras.optimizers.Adam(),
                        loss=tf.losses.SparseCategoricalCrossentropy(from_logits=True)
                        )

Model: "my_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ (64, 60, 256)               │       6,571,776 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm (LSTM)                          │ ((64, 60, 1024), (64,       │       5,246,976 │
│                                      │ 1024), (64, 1024))          │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (64, 60, 25671)             │      26,312,775 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 38,131,527 (145.46 MB)

 Trainable params: 38,131,527 (145.46 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
EPOCHS = 100
word2word_history = word2word_model.fit(w2w_dataset,
                                        epochs=EPOCHS,
                                        )

Epoch 1/100
51/51 ━━━━━━━━━━━━━━━━━━━━ 16s 302ms/step - loss: 4.6515
Epoch 2/100
51/51 ━━━━━━━━━━━━━━━━━━━━ 16s 306ms/step - loss: 4.5048
Epoch 3/100
51/51 ━━━━━━━━━━━━━━━━━━━━ 17s 317ms/step - loss: 4.3429
Epoch 4/100
51/51 ━━━━━━━━━━━━━━━━━━━━ 21s 323ms/step - loss: 4.2092
Epoch 5/100
51/51 ━━━━━━━━━━━━━━━━━━━━ 17s 334ms/step - loss: 4.0754
Epoch 6/100
51/51 ━━━━━━━━━━━━━━━━━━━━ 18s 341ms/step - loss: 3.9457
Epoch 7/100
51/51 ━━━━━━━━━━━━━━━━━━━━ 17s 333ms/step - loss: 3.8230
Epoch 8/100
51/51 ━━━━━━━━━━━━━━━━━━━━ 20s 327ms/step - loss: 3.7016
Epoch 9/100
51/51 ━━━━━━━━━━━━━━━━━━━━ 17s 333ms/step - loss: 3.5611
Epoch 10/100
51/51 ━━━━━━━━━━━━━━━━━━━━ 17s 336ms/step - loss: 3.4603
Epoch 11/100
51/51 ━━━━━━━━━━━━━━━━━━━━ 17s 335ms/step - loss: 3.3586
Epoch 12/100
51/51 ━━━━━━━━━━━━━━━━━━━━ 17s 332ms/step - loss: 3.2467
Epoch 13/100
51/51 ━━━━━━━━━━━━━━━━━━━━ 18s 331ms/step - loss: 3.1523
Epoch 14/100
51/51 ━━━━━━━━━━━━━━━━━━━━ 20s 331ms/step - loss: 3.0552
Epoch 15/100
51/51 ━━━━━━━━━━

In [ ]:
word2word_model.save('/content/drive/MyDrive/Aprendizaje 2/TP2/word2word_model.keras')

In [ ]:
# Guardar modelo
#word2word_model.save('/content/drive/MyDrive/Aprendizaje 2/TP2/word2word_model2.keras')

In [ ]:
# Cargar el modelo guardado
word2word_model = load_model('/content/drive/MyDrive/Aprendizaje 2/TP2/word2word_model2.keras',
                          custom_objects={'MyModel': MyModel})

### **Generación de texto palabra a palabra**

In [ ]:
class OneStepWord(tf.keras.Model):
    def __init__(self, model, words_from_ids, ids_from_words, temperature=1.0):
        super().__init__()
        self.temperature = temperature
        self.model = model
        self.words_from_ids = words_from_ids
        self.ids_from_words = ids_from_words

    @tf.function
    def generate_one_step(self, inputs, states=None):
        # Convertir cadenas a IDs de palabras
        input_words = tf.strings.split(inputs)
        input_ids = self.ids_from_words(input_words).to_tensor()

        # Ejecutar el modelo
        # predicted_logits.shape es [batch, word, next_word_logits]
        predicted_logits, states = self.model(inputs=input_ids, states=states,
                                              return_state=True)

        # Solo usar la última predicción
        predicted_logits = predicted_logits[:, -1, :]
        predicted_logits = predicted_logits / self.temperature

        # Muestrear los logits de salida para generar IDs de palabras
        predicted_ids = tf.random.categorical(predicted_logits, num_samples=1)
        predicted_ids = tf.squeeze(predicted_ids, axis=-1)

        # Convertir de IDs de palabras a palabras
        predicted_words = self.words_from_ids(predicted_ids)

        # Retornar las palabras y el estado del modelo
        return predicted_words, states

In [ ]:
# Función para imprimir el texto centrado en la consola
def print_centered(text, width=80):
    wrapped_text = textwrap.fill(text, width=width)
    lines = wrapped_text.split("\n")
    for line in lines:
        print(line.center(width))

# Función para generar fragmentos de texto con ajuste de impresión
def text_generator_w2w(model, temperature, seq_length, initial_imput="Oh Lord", num_fragments=2):
    # Lista para almacenar los fragmentos generados
    generated_fragments = []

    # Crear el modelo de generación con la temperatura específica
    one_step_model = OneStepWord(model, words_from_ids, ids_from_words, temperature=temperature)

    print(f"\n{'='*100}")
    print(f"TEMPERATURA: {temperature} | LONGITUD: {seq_length}".center(100))
    print(f"{'='*100}\n")

    for i in range(num_fragments):
        states = None
        next_word = tf.constant([initial_imput])
        result = [next_word]

        # Generar texto con la longitud especificada
        for _ in range(seq_length):
            next_word, states = one_step_model.generate_one_step(next_word, states=states)
            result.append(next_word)

        # Convertir a una sola cadena correctamente
        text_new = tf.strings.join(result, separator=" ").numpy()[0].decode('utf-8')
        generated_fragments.append((temperature, seq_length, text_new))

        # Obtener el ancho de la terminal
        terminal_width = shutil.get_terminal_size().columns

        # Mostrar el fragmento generado con centrado
        print(f"\n{'-'*terminal_width}")
        print(f"FRAGMENTO {i+1}".center(terminal_width))
        print(f"{'-'*terminal_width}\n")
        print_centered(text_new, width=min(terminal_width, 100))

In [ ]:
# Primer generación de texto
text_generator_w2w(word2word_model, 1.1, 75, "The", 10)


                                  TEMPERATURA: 1.1 | LONGITUD: 75                                   


----------------------------------------------------------------------------------------------------
                                            FRAGMENTO 1                                             
----------------------------------------------------------------------------------------------------

 The trick of's frown, his forehead, nay, the valley, The pretty dimples of his chin and cheek, His 
 smiles, The very mould and frame of hand, nail, finger: And thou, good goddess Nature, which hast  
 made it less To prick his looks and us; In him that made us we need to answer it. When he's a dear 
     grown by his life is the worst thing he in the air. QUEEN MARGARET: Hover about her; say,      

----------------------------------------------------------------------------------------------------
                                            FRAGMENTO 2                               

In [ ]:
# Segunda generación de texto
text_generator_w2w(word2word_model, 1.0, 50)


                                  TEMPERATURA: 1.0 | LONGITUD: 50                                   


----------------------------------------------------------------------------------------------------
                                            FRAGMENTO 1                                             
----------------------------------------------------------------------------------------------------

Oh Lord You're hath more; And nothing can my name might well enjoy it, and we'll to Brittany. Come, 
   therefore, let's about it speedily. KING HENRY VI: Were he not for a piece of virtue, see poor   
              mother, May meet with him, Whose way we may enter in, To the winds whose              

----------------------------------------------------------------------------------------------------
                                            FRAGMENTO 2                                             
--------------------------------------------------------------------------------------

In [ ]:
# Tercera generación de texto
text_generator_w2w(word2word_model, 1.0, 100, "Night")


                                  TEMPERATURA: 1.0 | LONGITUD: 100                                  


----------------------------------------------------------------------------------------------------
                                            FRAGMENTO 1                                             
----------------------------------------------------------------------------------------------------

Night behold. fair, known talk, eyes drop eyes are dangerous and your from him, I may be married to 
 my liege, I beseech your grace good rest! Sorrow breaks seasons and reposing hours, their more has 
 sworn but not all The grave. Why, what said the life is past at home by the other lose, that made  
King Richard that himself hunt about him. O, then; I'll go alone. KING RICHARD II: Give me thy hand,
 Kate: we will not stay. DUCHESS OF YORK: Hadst thou groan'd for grace, And set thy diadem upon my  
                                  head; Or bide the mortal fortune                     

In [ ]:
# Cuarta generación de texto
text_generator_w2w(word2word_model, 1.5, 200, "My heart")


                                  TEMPERATURA: 1.5 | LONGITUD: 200                                  


----------------------------------------------------------------------------------------------------
                                            FRAGMENTO 1                                             
----------------------------------------------------------------------------------------------------

 My heart nor fears Benvolio say he ne'er would not. Your hand: which God my word for-- GLOUCESTER: 
 Romeo! my daughter, early fellow, Never I help on Edward be as like To lose this practise so, But  
 who not, great Romeo, Warwick was too very kind, and true To do him good? KING LEWIS XI: Welcome,  
 Thomas devise! take the life against me; For the inheritance of his divided because your son, Whom 
 first from my sweet dishonour to tell. KING HENRY VI: And long nor love with mine, That there draw 
 us And himself to death. Of what I'll try in we call him comforts. Sir, The manner of 

## **Análisis de fragmentos**



**Segmentos seleccionados del modelo caracter a caracter**

*  Primera generación de texto | Temperatura: 1.3 | Longitud: 50


    Fragmento 6:

    The truth will be penished in this tune. (La verdad será castigada en esta melodía.)

    BASTHASAR:

    I will


    Fragmento 7:

    The truth will set again with crowns. (La verdad volverá a fijarse con coronas.)

    BUCKINGHAM:

    Why, the

    Fragmento 10:
    The truth common people had embraced me in (La verdad es que la gente común me había abrazado en)
    so far from henc

*  Segunda generación de texto | Temperatura: 0.5 | Longitud: 50


    Fragmento 2:
    The love shall be death disgraced thy daughter. (El amor será la muerte deshonrada de tu hija.)
    What, will

*  Tercera generación de texto | Temperatura: 1 | Longitud: 100


    Fragmento 2:
    Oh Lord Hastings, let us hear him pass. (Oh Lord Hastings, oigámosle pasar.)

    Second Servingman:
    Ay, and his himself, but madam: ha! (Ay, y el suyo, pero señora: ¡ja!)

    AUTOLYCUS:

**Observaciones:**

En el casos del modelo de generación de texto caracter a caracter, el corpus generado conserva el formato de poesía, generando parrafos, saltos de lineas y estructuras propias de la poesia. En temperaturas cercanas a 1, la generación parece tener más sentido, ademas, los segmentos generados parecen guardar más sentidos en longitudes más cortas.



---



**Segmentos seleccionados del modelo palabra a palabra**

* Primera generación | Temperatura: 1.1 | Longitud: 75

      Fragmetno 1

      The trick of's frown, his forehead, nay, the valley, The pretty dimples of his chin and cheek, His smiles, The very mould and frame of hand, nail, finger: And thou, good goddess Nature, which hast made it less To prick his looks and us; In him that made us we need to answer it. When he's a dear grown by his life is the worst thing he in the air. QUEEN MARGARET: Hover about her; say,
      (El truco de su ceño, su frente, mejor dicho, el valle, Los bonitos hoyuelos de su barbilla y mejilla, Sus sonrisas, El mismo molde y estructura de la mano, la uña, el dedo: Y tú, buena diosa Naturaleza, que has hecho que sea menos para irritar sus miradas y a nosotros; En aquel que nos hizo debemos responderla. Cuando es un querido adulto su vida es lo peor que hay en el aire. REINA MARGARITA: Pasa el cursor sobre ella; decir,)

      Fragmento 5

      The nature of their crimes, that I more might have said, but against the other fortune of the world. This news is old enough, yet it is least expected. ISABELLA: Ho, by your leave! POLIXENES: By heaven, We have a son for love a very pretty boy. O' my troth, I looked upon him o' Wednesday half an hour together: has such a confirmed countenance. I saw him run and full as beauty, As just, brave
      (La naturaleza de sus crímenes, eso podría haber dicho mejor, pero contra la otra fortuna del mundo. Esta noticia es bastante antigua, pero es la menos esperada. ISABELLA: ¡Hola, con tu permiso! POLIXENES: Por el cielo, tenemos un hijo por amor, un muchacho muy lindo. A fe mía, lo miré el miércoles durante media hora seguida: tiene un semblante tan confirmado. Lo vi correr y lleno como belleza, como justo, valiente)

      Fragmento 8

      The trick of's frown, his forehead, nay, the valley, The pretty dimples of his chin and cheek, His smiles, The Conspirator: Your answer Henry as his new plants with dews of flattery, Seducing so my friends; and, to this end, now I'll pine or think thou thee, then am I hope to thee, Than with this knife thou his most royal root, Is crack'd, and all the one half of my heart; And if you crown
      (El truco de su ceño, su frente, no, el valle, Los bonitos hoyuelos de su barbilla y mejilla, Sus sonrisas, El Conspirador: Tu respuesta Henry como sus nuevas plantas con rocío de adulación, Seduciendo así a mis amigos; y, con este fin, ahora te añoraré o pensaré en ti, entonces espero en ti que con este cuchillo tu raíz más real sea partida, y toda la mitad de mi corazón; Y si coronas)

      Fragmento 9

      The wanton Edward, and the lusty George? And where's that valiant crook-back prodigy, Dicky your fearful and unpitied end: Earth gapes, hell burns, fiends roar, saints pray. To me, I'll hate him everlastingly in the table. Second Gentleman: 'Thou shalt thou do in heart Of what I was a bride.  
      PARIS: Younger than to me; and with speed so pace of him that had said married his wounds doth down, And buried such a confirmed countenance.
      (¿El lascivo Edward y el lujurioso George? ¿Y dónde está ese valiente prodigio ladrón, Dicky, tu final temeroso y despiadado: la tierra se abre, el infierno arde, los demonios rugen, los santos rezan? Para mí, lo odiaré eternamente en la mesa. Segundo Caballero: 'Harás de corazón lo que yo fui novia.  
      PARÍS: Más joven que yo; y con la velocidad, el paso del que había dicho casado, sus heridas bajan, y entierra un semblante tan confirmado.)

*  Segunda generación | Temperatura: 1.0 | Longitud: 50

        Fragmento 2

        Oh Lord Plantagenet, caitiff tears she came from this princely presence. Now, coming me to your grace's hands. GLOUCESTER: What, shall I groan and tell me the very dog seems to a grace so noble Vilely bound up? What would he say? Or how Should be his day, Go home thy back
        (Oh Señor Plantagenet, lágrimas de caitiff brotaron de esta presencia principesca. Ahora, llegando a las manos de vuestra majestad. GLOUCESTER: ¿Qué? ¿Debo gemir y decirme que el mismo perro le parece a una gracia tan noble Vilmente atado? ¿Qué diría? O cómo debería ser su día, vete a casa de espaldas.)

Observaciones:

En este caso, en el modelo de generación de texto palabra a palabra, no se conservó la estructura propia de una poesía. Los fragmentos generados presentan mayor coherencia en con temperaturas cercanas a 1, es decir más neutros. En cuanto a las longitudes, en secuencias más cortas hay un mejor resultado.

## **Conclusión**

Probé dos modelos de redes recurrentes para generar texto: uno que predice carácter a carácter y otro palabra a palabra. Aunque en teoría el modelo palabra a palabra debería generar frases más coherentes, en la práctica el modelo carácter a carácter funcionó mejor.

Ambos modelos usaron un preprocesamiento similar, pero el de carácter a carácter conservó mejor la estructura del texto, respetando la puntuación y los saltos de línea. En cambio, el modelo palabra a palabra perdió esta información al hacer el split, lo que podría solucionarse con expresiones regulares que mantengan la estructura original.

En cuanto a la coherencia, el modelo palabra a palabra generó segmentos con cierta fluidez dentro de cada parte del texto, aunque sin una conexión global clara. Funcionó mejor con temperaturas más neutras y secuencias cortas.

Si bien ambos modelos lograron generar texto, la simplicidad de las arquitecturas limitó la calidad de los resultados. La incorporación de capas que analicen la secuencia en ambas direcciones podría ayudar a generar textos más estructurados y naturales.